In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# India YouTube Top Channels (Current Dataset) - EDA
Simple analysis for ranking, subscribers, views, and channel distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

# Works both on Kaggle and local
candidate_paths = [
    '/kaggle/input/youtube-top-india-by-subscribers/youtube_top_india_by_subscribers.csv',
    '/kaggle/input/datasets/youtube_top_india_by_subscribers.csv',
    './youtube_top_india_by_subscribers.csv'
]

DATA_PATH = None
for p in candidate_paths:
    if Path(p).exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    # Fallback: search by filename under /kaggle/input
    for root, _, files in os.walk('/kaggle/input'):
        if 'youtube_top_india_by_subscribers.csv' in files:
            DATA_PATH = os.path.join(root, 'youtube_top_india_by_subscribers.csv')
            break

if DATA_PATH is None:
    raise FileNotFoundError('Could not find youtube_top_india_by_subscribers.csv')

df = pd.read_csv(DATA_PATH)

# Normalize columns
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

print('Loaded from:', DATA_PATH)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(3))

In [ ]:
# Data quality
display(df.info())

quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique()
}).sort_values('missing_values', ascending=False)

display(quality)
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Convert metrics
for col in ['rank', 'subscribers', 'views']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if {'views', 'subscribers'}.issubset(df.columns):
    df['views_per_subscriber'] = np.where(df['subscribers'] > 0, df['views'] / df['subscribers'], np.nan)

display(df.describe(include='all').T)

In [ ]:
# Top 10 channels by subscribers
top10 = df.nlargest(10, 'subscribers')[['title', 'subscribers']].copy()

# Safe display labels for environments with limited font support
top10['title_plot'] = top10['title'].astype(str).str.encode('ascii', errors='ignore').str.decode('ascii').str.strip()
top10['title_plot'] = top10['title_plot'].replace('', np.nan).fillna('Non-Latin Channel')

display(top10[['title', 'subscribers']])

plt.figure(figsize=(10, 5))
sns.barplot(data=top10, x='subscribers', y='title_plot', hue='title_plot', palette='Blues_r', dodge=False, legend=False)
plt.title('Top 10 Channels by Subscribers')
plt.xlabel('Subscribers')
plt.ylabel('Channel')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['subscribers'].dropna(), bins=25, ax=axes[0], color='teal')
axes[0].set_title('Subscribers Distribution')

sns.histplot(df['views'].dropna(), bins=25, ax=axes[1], color='orange')
axes[1].set_title('Views Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Rank relationships
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=df, x='rank', y='subscribers', alpha=0.7, ax=axes[0])
axes[0].set_title('Rank vs Subscribers')

sns.scatterplot(data=df, x='rank', y='views', alpha=0.7, ax=axes[1], color='darkorange')
axes[1].set_title('Rank vs Views')

for ax in axes:
    ax.invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Most efficient channels by views per subscriber
if 'views_per_subscriber' in df.columns:
    vps = df.nlargest(10, 'views_per_subscriber')[['title', 'views_per_subscriber']]
    display(vps)
else:
    print('views_per_subscriber column not found.')

## Summary
- Checked shape, quality, and missing values.
- Reviewed top channels by subscribers.
- Visualized subscribers/views distributions and rank patterns.
- Compared channels by views per subscriber.